In [1]:
import os
import numpy as np
import librosa
import pandas as pd

# Basic Audio Settings
SR = 22050
BASE_DIR = "/home/feliciano/LOBSTER SOUNDS/DatasetClass"
CLASS_FOLDERS = {
    "Adult": "adult_lobsters",
    "Juvenile": "juvenile_lobsters",
    "Male": "male_lobsters",
    "Female": "female_lobsters"
}

def analyze_acoustic_features(folder_path):
    features = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
    
    # Analyze up to 50 files per group to get a solid average
    for f in files[:50]:
        path = os.path.join(folder_path, f)
        y, sr = librosa.load(path, sr=SR)
        
        # 1. Peak Frequency (Dominant Frequency)
        fft = np.abs(np.fft.rfft(y))
        freqs = np.fft.rfftfreq(len(y), 1/sr)
        peak_freq = freqs[np.argmax(fft)]
        
        # 2. Spectral Centroid (Brightness/Gravity)
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        
        # 3. Spectral Bandwidth (Spread of frequencies)
        bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        
        # 4. RMS Energy
        rms = np.mean(librosa.feature.rms(y=y))
        
        features.append([peak_freq, centroid, bandwidth, rms])
        
    return np.mean(features, axis=0)

results = []
for label, folder in CLASS_FOLDERS.items():
    full_path = os.path.join(BASE_DIR, folder)
    if os.path.exists(full_path):
        metrics = analyze_acoustic_features(full_path)
        results.append([label] + list(metrics))

# Create Summary Table
df = pd.DataFrame(results, columns=['Group', 'Peak Freq (Hz)', 'Centroid (Hz)', 'Bandwidth (Hz)', 'Energy (RMS)'])
print("\n--- ACOUSTIC CHARACTERISTICS PER GROUP ---")
print(df.to_string(index=False))


--- ACOUSTIC CHARACTERISTICS PER GROUP ---
   Group  Peak Freq (Hz)  Centroid (Hz)  Bandwidth (Hz)  Energy (RMS)
   Adult          179.60    1141.415215     1545.515957      0.216579
Juvenile          102.00    1070.526550     1475.326683      0.440403
    Male          119.04    1158.673258     1555.034944      0.226117
  Female          119.76    1043.468167     1465.472610      0.335163
